# 04 — Adversarial Drift Simulation

**Threat model**: fraudsters observe that high-value transactions are flagged, so they split each transaction into micro-amounts (≤$50) spread across time — suppressing both the `TransactionAmt` and `velocity_value` signals.

We simulate this by modifying the test set and re-scoring with the fixed model, then measuring:
- Recall drop
- Population Stability Index (PSI) on key features — alarm at PSI > 0.2

Output: appended to `data_export.json`

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, recall_score, precision_recall_curve

DATA_DIR    = Path("../../..") / "data" / "ieee-fraud-detection"
DEMO_DIR    = Path("..") / "demo"
EXPORT_PATH = DEMO_DIR / "data_export.json"
MODEL_PATH  = Path("../../..") / "models" / "fraud_xgb_calibrated.pkl"

EVASION_CAP  = 50.0   # fraudsters cap amounts to ≤$50
VELOCITY_CAP = 80.0   # and suppress velocity value signal
PSI_ALARM    = 0.2


## 1. Load model

In [ ]:
with open(MODEL_PATH, "rb") as f:
    bundle = pickle.load(f)

xgb_base      = bundle["xgb_base"]
iso           = bundle["iso"]
feature_names = bundle["feature_names"]

def score(X):
    return iso.predict(xgb_base.predict_proba(X)[:, 1])

print(f"Model loaded — {len(feature_names)} features")


## 2. Reconstruct test feature matrix

In [ ]:
df_pred = pd.read_parquet(DATA_DIR / "test_predictions.parquet")
eng     = pd.read_parquet(DATA_DIR / "train_engineered.parquet")
eng_test = (
    eng[eng["TransactionID"].isin(df_pred["TransactionID"])]
    .set_index("TransactionID")
    .loc[df_pred["TransactionID"].values]
    .reset_index()
)

drop_cols = ["TransactionID","TransactionDT","isFraud",
             "card2","card3","card5","addr1","addr2",
             "P_emaildomain","R_emaildomain","DeviceInfo","DeviceType",
             "card4","card6","ProductCD",
             "M1","M2","M3","M4","M5","M6","M7","M8","M9",
             "id_12","id_15","id_16","id_23","id_27","id_28",
             "id_29","id_30","id_31","id_32","id_33","id_34",
             "id_35","id_36","id_37","id_38"]
drop_cols = [c for c in drop_cols if c in eng_test.columns]

X_test = eng_test.drop(columns=drop_cols)
for col in X_test.select_dtypes("object").columns:
    X_test[col] = X_test[col].astype("category").cat.codes
for col in feature_names:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[feature_names]

y_true = eng_test["isFraud"].values
print(f"Test set: {len(X_test):,} rows  |  fraud: {y_true.mean():.2%}")


## 3. Baseline scoring

In [ ]:
with open(EXPORT_PATH) as f:
    export = json.load(f)
threshold = export.get("optimal_threshold", 0.5)

y_prob_baseline  = score(X_test)
baseline_recall  = recall_score(y_true, (y_prob_baseline >= threshold).astype(int), zero_division=0)
baseline_prauc   = average_precision_score(y_true, y_prob_baseline)

print(f"Threshold       : {threshold:.3f}")
print(f"Baseline recall : {baseline_recall:.3f}")
print(f"Baseline PR-AUC : {baseline_prauc:.4f}")


## 4. Apply adversarial shift

Apply amount capping to all fraud rows + 30% of legitimate rows to simulate a population-level distribution shift (so PSI is detectable on the monitoring dashboard).

In [ ]:
rng        = np.random.default_rng(42)
fraud_mask = y_true == 1
legit_shift = (~fraud_mask) & (rng.random(len(y_true)) < 0.30)
shift_mask  = fraud_mask | legit_shift

print(f"Rows shifted: {shift_mask.sum():,}  ({shift_mask.mean():.1%} of test set)")
print(f"  Fraud rows   : {fraud_mask.sum():,}  (all)")
print(f"  Legit rows   : {legit_shift.sum():,}  (30% sample)")


## 5. Re-score on shifted data

In [ ]:
X_shifted = X_test.copy()
X_shifted.loc[shift_mask, "TransactionAmt"] = (
    X_shifted.loc[shift_mask, "TransactionAmt"].clip(upper=EVASION_CAP)
)
if "velocity_value_1h" in X_shifted.columns:
    X_shifted.loc[shift_mask, "velocity_value_1h"] = (
        X_shifted.loc[shift_mask, "velocity_value_1h"].clip(upper=VELOCITY_CAP)
    )
if "velocity_value_24h" in X_shifted.columns:
    X_shifted.loc[shift_mask, "velocity_value_24h"] = (
        X_shifted.loc[shift_mask, "velocity_value_24h"].clip(upper=VELOCITY_CAP * 5)
    )

y_prob_shifted = score(X_shifted)
shifted_recall = recall_score(y_true, (y_prob_shifted >= threshold).astype(int), zero_division=0)
shifted_prauc  = average_precision_score(y_true, y_prob_shifted)

print(f"Post-drift recall : {shifted_recall:.3f}  (was {baseline_recall:.3f})")
print(f"Recall drop       : {baseline_recall - shifted_recall:.3f}  ({(baseline_recall-shifted_recall)/baseline_recall:.1%} relative)")
print(f"Post-drift PR-AUC : {shifted_prauc:.4f}  (was {baseline_prauc:.4f})")


## 6. Population Stability Index (PSI)

PSI measures how much a distribution has shifted between baseline and current. PSI > 0.2 triggers a drift alarm — time to investigate or retrain.

We use **quantile-based bins** rather than uniform bins, which is more sensitive for skewed financial distributions.

In [ ]:
def compute_psi(baseline, shifted, bins=10):
    edges = np.unique(np.percentile(baseline, np.linspace(0, 100, bins + 1)))
    if len(edges) < 3:
        return 0.0
    base_cnt,  _ = np.histogram(baseline, bins=edges)
    shift_cnt, _ = np.histogram(shifted,  bins=edges)
    base_pct  = (base_cnt  + 1e-6) / len(baseline)
    shift_pct = (shift_cnt + 1e-6) / len(shifted)
    return float(np.sum((shift_pct - base_pct) * np.log(shift_pct / base_pct)))

psi_amount   = compute_psi(X_test["TransactionAmt"].values,   X_shifted["TransactionAmt"].values)
psi_velocity = compute_psi(X_test["velocity_value_1h"].values, X_shifted["velocity_value_1h"].values)

print(f"PSI — TransactionAmt    : {psi_amount:.4f}  {'⚠ ALARM' if psi_amount > PSI_ALARM else '✓ OK'}  (alarm > {PSI_ALARM})")
print(f"PSI — velocity_value_1h : {psi_velocity:.4f}  {'⚠ ALARM' if psi_velocity > PSI_ALARM else '✓ OK'}")


## 7. Export results

In [ ]:
prec_s, rec_s, thresh_s = precision_recall_curve(y_true, y_prob_shifted)
thresh_s = np.append(thresh_s, 1.0)
pr_curve_shifted = [
    {"threshold": round(float(t),4), "precision": round(float(p),4), "recall": round(float(r),4)}
    for t, p, r in zip(thresh_s, prec_s, rec_s)
]

export["adversarial"] = {
    "strategy":        "transaction_splitting",
    "evasion_cap":     EVASION_CAP,
    "baseline_recall": round(baseline_recall, 4),
    "shifted_recall":  round(shifted_recall, 4),
    "recall_drop":     round(baseline_recall - shifted_recall, 4),
    "baseline_prauc":  round(baseline_prauc, 4),
    "shifted_prauc":   round(shifted_prauc, 4),
    "psi_amount":      round(psi_amount, 4),
    "psi_velocity":    round(psi_velocity, 4),
    "psi_alarm_threshold": PSI_ALARM,
    "pr_curve_shifted": pr_curve_shifted,
}

with open(EXPORT_PATH, "w") as f:
    json.dump(export, f)

print(f"Saved → {EXPORT_PATH}")
